In [1]:
from datasets import load_dataset
import json
path="VLFFD-main/output.json"
data = load_dataset("json", data_files=path, split="train")

data = data.train_test_split(
    train_size=0.9, test_size=0.1, seed=42, shuffle=True
)
data["validation"] = data.pop("test")
data

DatasetDict({
    train: Dataset({
        features: ['question', 'imagepath', 'answer', 'reason'],
        num_rows: 3870
    })
    validation: Dataset({
        features: ['question', 'imagepath', 'answer', 'reason'],
        num_rows: 430
    })
})

In [ ]:
data["train"][0]

For this classification task, create a multiple-choice question prompt and preprocess the data into a multimodal conversational format.

In [ ]:
from typing import Any
from PIL import Image
def format_data(example:dict[str,Any])-> dict[str, Any]:
    example["messages"]=[
        {
            "role": "user", 
            "content": [
                {
                    "type":"image",
                },
                {
                    "type":"text",
                    "text":example["question"],
                },
            ],
        },
        {
            "role":"assistant",
            "content":[
                {
                    "type":"text",
                    "text":example["reason"]+'\n'+example["answer"],
                },
            ],
        },
    ]
    example["image"] = Image.open(example["imagepath"])
    return example
data = data.map(format_data, remove_columns=["question","imagepath","reason"])
data["train"][0]

## Fine-tune the model with LoRA
Traditional fine-tuning of large language models is resource-intensive because it requires adjusting billions of parameters. Parameter-Efficient Fine-Tuning (PEFT) addresses this by training a smaller number of parameters. A common PEFT technique is Low-Rank Adaptation (LoRA), which efficiently adapts large language models by training small, low-rank matrices that are added to the original model instead of updating the full-weight matrices. In QLoRA, the base model is quantized to 4-bit before its weights are frozen, then LoRA adapter layers are attached and trained.

This notebook demonstrates supervised fine-tuning MedGemma with QLoRA using the SFTTrainer from the Hugging Face TRL library.

In [ ]:
import torch
from transformers import AutoProcessor,AutoModelForImageTextToText
from transformers.utils.quantization_config import BitsAndBytesConfig

torch.set_default_device("mps")
model_kwargs = dict(
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
# quantization_config = BitsAndBytesConfig(
#     load_in_8bit=True,
#     bnb_8bit_use_double_quant=True,
#     bnb_8bit_quant_type="nf4",
#     bnb_8bit_compute_dtype=model_kwargs["torch_dtype"],
#     bnb_8bit_quant_storage=model_kwargs["torch_dtype"],
# )
model_id="google/gemma-3-4b-it"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(model_id, **model_kwargs)
# model = AutoModelForImageTextToText.from_pretrained(model_id, **model_kwargs)
# Use right padding to avoid issues during training
processor.tokenizer.padding_side = "right"

Create a LoraConfig. It will be provided to the SFTTrainer, which supports built-in integration with the Hugging Face PEFT library.

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=[
        "lm_head",
        "embed_tokens",
    ],
)

Define a custom data collator that processes examples containing text and images and returns batches of data in the expected model input format.

In [ ]:
from typing import Any

def collate_fn(examples: list[dict[str, Any]]):
    texts = []
    images = []
    for example in examples:
        images.append([example["image"].convert("RGB")])
        texts.append(processor.apply_chat_template(
            example["messages"], add_generation_prompt=False, tokenize=False
        ).strip())

    # Tokenize the texts and process the images
    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

    # The labels are the input_ids, with the padding and image tokens masked in
    # the loss computation
    labels = batch["input_ids"].clone()

    # Mask image tokens
    image_token_id = [
        processor.tokenizer.convert_tokens_to_ids(
            processor.tokenizer.special_tokens_map["boi_token"]
        )
    ]
    # Mask tokens that are not used in the loss computation
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == image_token_id] = -100
    labels[labels == 262144] = -100

    batch["labels"] = labels
    return batch

In [ ]:
from trl import SFTConfig

num_train_epochs = 1  # @param {type: "number"}
learning_rate = 2e-4  # @param {type: "number"}

args = SFTConfig(
    output_dir="./output",            # Directory and Hub repository id to save the model to
    num_train_epochs=num_train_epochs,                       # Number of training epochs
    per_device_train_batch_size=2,                           # Batch size per device during training
    per_device_eval_batch_size=2,                            # Batch size per device during evaluation
    gradient_accumulation_steps=2,                           # Number of steps before performing a backward/update pass
    gradient_checkpointing=True,                             # Enable gradient checkpointing to reduce memory usage
    optim="adamw_torch_fused",                               # Use fused AdamW optimizer for better performance
    logging_steps=50,                                        # Number of steps between logs
    save_strategy="epoch",                                   # Save checkpoint every epoch
    eval_strategy="steps",                                   # Evaluate every `eval_steps`
    eval_steps=50,                                           # Number of steps between evaluations
    learning_rate=learning_rate,                             # Learning rate based on QLoRA paper
    # bf16=True,                                               # Use bfloat16 precision
    max_grad_norm=0.3,                                       # Max gradient norm based on QLoRA paper
    warmup_ratio=0.03,                                       # Warmup ratio based on QLoRA paper
    lr_scheduler_type="linear",                              # Use linear learning rate scheduler
    push_to_hub=False,                                        # Push model to Hub
    report_to="tensorboard",                                 # Report metrics to tensorboard
    gradient_checkpointing_kwargs={"use_reentrant": False},  # Set gradient checkpointing to non-reentrant to avoid issues
    dataset_kwargs={"skip_prepare_dataset": True},           # Skip default dataset preparation to preprocess manually
    remove_unused_columns = False,                           # Columns are unused for training but needed for data collator
    label_names=["labels"],                                  # Input keys that correspond to the labels
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=data["train"],
    eval_dataset=data["validation"].shuffle().select(range(200)),  # Use subset of validation set for faster run
    peft_config=peft_config,
    processing_class=processor,
    data_collator=collate_fn,
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model()